Train Weekly Model — Live Production Data
Trains LightGBM on weekly gold data, evaluates against baseline, logs to MLflow, saves next-week prediction.

**Input**: gold/erp/battery/phase1_overall_weekly_live.parquet
**Output**: MLflow logged model + printed next-week forecast

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_gold
import pandas as pd
import mlflow
import mlflow.lightgbm
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

blob_service = get_blob_service(storage_account_name, storage_account_key)

def wape(y_true, y_pred):
    return abs(y_true - y_pred).sum() / abs(y_true).sum()

In [0]:
gold_weekly = read_gold(blob_service, "live/battery/data/phase1_overall_weekly_live.parquet")
gold_weekly["week_start"] = pd.to_datetime(gold_weekly["week_start"])

feature_cols = ["week_of_year", "month", "contains_month_end", "lag_4w", "rolling_avg_4w"]
target_col = "total_units_sold"

model_data = gold_weekly.dropna(subset=feature_cols + [target_col]).copy()
model_data = model_data.sort_values("week_start")

split_idx = int(len(model_data) * 0.8)
train_data = model_data.iloc[:split_idx]
test_data = model_data.iloc[split_idx:]

print(f"Train: {len(train_data)} weeks, Test: {len(test_data)} weeks")

X_train, y_train = train_data[feature_cols], train_data[target_col]
X_test, y_test = test_data[feature_cols], test_data[target_col]

In [0]:
with mlflow.start_run(run_name="phase1_overall_weekly_live"):
    params = {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 6}
    mlflow.log_params(params)

    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    wape_score = wape(y_test, preds)

    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("wape", wape_score)
    mlflow.lightgbm.log_model(model, name="model")

    print(f"Model — MAE: {mae:.2f}   RMSE: {rmse:.2f}   WAPE: {wape_score:.3%}")

    baseline_preds = X_test["rolling_avg_4w"]
    print(f"Baseline — MAE: {mean_absolute_error(y_test, baseline_preds):.2f}   WAPE: {wape(y_test, baseline_preds):.3%}")

In [0]:
final_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
X_all = model_data[feature_cols]
y_all = model_data[target_col]
final_model.fit(X_all, y_all)

last_week_start = gold_weekly["week_start"].max()
next_week_start = last_week_start + pd.Timedelta(weeks=1)
next_week_end = next_week_start + pd.Timedelta(days=6)

lag_4w_value = gold_weekly[gold_weekly["week_start"] == next_week_start - pd.Timedelta(weeks=4)]["total_units_sold"].values
lag_4w_value = lag_4w_value[0] if len(lag_4w_value) > 0 else None

next_week_features = pd.DataFrame([{
    "week_of_year": next_week_start.isocalendar()[1],
    "month": next_week_start.month,
    "contains_month_end": int(next_week_start.month != next_week_end.month),
    "lag_4w": lag_4w_value,
    "rolling_avg_4w": gold_weekly["total_units_sold"].tail(4).mean(),
}])

next_week_pred = final_model.predict(next_week_features[feature_cols])
print(f"Predicted net units for week starting {next_week_start.date()}: {next_week_pred[0]:.0f}")